# NCERT QA - Difficulty Assignment (LLM-as-a-judge)

Assigns `difficulty` (Easy / Medium / Hard) to 500 sampled NCERT QA rows.

**Task.** `question` is an NCERT textbook question (Hindi literature, social
science, history, civics), `answer` is the reference answer. Answers are
free-form prose - median 28 words - so there is no string to match against.

**Why a judge and not ROUGE-L.** These answers can be entirely correct while
sharing almost no n-grams with the reference, especially the Hindi literature
ones. Measured on this file, simply **echoing the question back scores 26%
ROUGE-L** - plausibly more than a correct answer worded differently. That
inversion is what a judge fixes.

**Pipeline.**

1. Three generators answer each question: **Mistral-7B**, **Llama-3.1-8B**,
   **Gemma-2-9B**.
2. An independent judge, **Qwen3-14B**, grades each answer against the
   reference on a 0-4 rubric.
3. A threshold turns each grade into a pass/fail vote, and the votes sum:

| Models passing | Difficulty |
|---|---|
| 3 / 3 | Easy |
| 2 / 3 | Medium |
| 0-1 / 3 | Hard |

**The judge is deliberately not one of the generators.** A model grading its
own output scores it higher - self-preference bias - which would inflate that
model's votes and skew every difficulty label. Qwen3-14B is a different family
from all three (Qwen vs Mistral / Meta / Google), larger than all three, and
ungated.

**Grades are read from logits, not generated.** The judge's rubric digit is
picked by argmax over the token ids for `0`-`4` in a single forward pass. It
cannot produce unparseable output, it is far faster than generating, and the
probability distribution over digits gives a smooth 0-1 score - which is what
makes re-thresholding free.

**Output.** `ncert_qa_difficulty.jsonl` - the 14 schema fields with
`difficulty` filled in and `eval_metric` set to `llm_as_a_judge`, plus an audit
file holding every answer and grade.

### Cell 1 - Install dependencies and authenticate

**An HF token is required.** Llama-3.1 and Gemma-2 are gated. Mistral and
Qwen3 are not. Accept the two licences on huggingface.co, create a **read**
token, then add it in Colab via the **key icon** as a secret named `HF_TOKEN`.
Use the secret rather than pasting the token into a cell.

In [1]:
!pip -q install -U transformers accelerate bitsandbytes huggingface_hub

HF_OK = False
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"))
    HF_OK = True
    print("HF login OK")
except Exception as e:
    print("No HF token ({}: {})".format(type(e).__name__, e))
    print("Mistral and Qwen3 still work; gated Llama/Gemma will fail with a 401.")

HF login OK


### Cell 2 - Mount Drive

Drive is the weight cache: each model is downloaded once, quantised to 4-bit,
and saved here, so later runs skip the download entirely. That now covers the
**judge as well as the three generators**.

**Watch your Drive quota.** Four cached models is roughly:

| Model | 4-bit on Drive |
|---|---|
| Mistral-7B | ~4.5 GB |
| Llama-3.1-8B | ~5.5 GB |
| Gemma-2-9B | ~6.5 GB |
| Qwen3-14B (judge) | ~9 GB |
| **Total** | **~25 GB** |

A free Google account has 15 GB total, so all four will not fit. The cell
prints your free space. If you are short, cache the judge and drop one
generator's cache, or re-download the smallest model each session.

In [2]:
import os

DRIVE_OK    = False
DRIVE_MOUNT = "/drive"
CACHE_DIR   = os.path.join(DRIVE_MOUNT, "MyDrive", "models")

try:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT, force_remount=True)
    DRIVE_OK = os.path.isdir(os.path.join(DRIVE_MOUNT, "MyDrive"))
except ImportError:
    print("Not running on Colab - Drive caching disabled.")
except Exception as e:
    print("DRIVE MOUNT FAILED: {}".format(e))

if DRIVE_OK:
    os.makedirs(CACHE_DIR, exist_ok=True)
    print("Drive mounted | weight cache: {}".format(CACHE_DIR))
    try:
        st = os.statvfs(DRIVE_MOUNT)
        free = st.f_bavail * st.f_frsize / 1e9
        print("free on Drive: {:.1f} GB".format(free))
        if free < 25:
            print("  NOTE: caching all four models needs ~25 GB. You have less,")
            print("  so some will re-download each session. That still works.")
    except Exception:
        pass
else:
    print("\nWARNING: no Drive - nothing is cached between sessions.")

Mounted at /drive
Drive mounted | weight cache: /drive/MyDrive/models
free on Drive: 10.6 GB
  NOTE: caching all four models needs ~25 GB. You have less,
  so some will re-download each session. That still works.


### Cell 3 - Configuration

- `GENERATORS` - the three models that answer the questions.
- `JUDGE` - **Qwen3-14B**, independent of all three. Ungated, ~9 GB in 4-bit,
  fits a T4 with room for the KV cache.
- `RUBRIC` - the 0-4 grading scale. Grades are stored raw; Cell 11 applies the
  threshold, so re-thresholding never re-runs a model.
- `THRESHOLD_MODE` - `"fixed"` (default, `0.625` = grade 2.5 of 4, i.e. better
  than "partially correct") or `"auto_median"`.
- `SET_EVAL_METRIC` - written into the output rows. Defaults to
  `llm_as_a_judge`, replacing the inherited `rouge_l`.

In [3]:
import gc
import re
import json
import random
import shutil
import statistics
from collections import Counter

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ---- paths ----
INPUT_FILE  = "ncert_qa.jsonl"
OUTPUT_FILE = "ncert_qa_difficulty.jsonl"
AUDIT_FILE  = "ncert_qa_audit.jsonl"
GEN_DIR     = "gen_progress"       # one file per generator
JUDGE_FILE  = "judge_progress.jsonl"

# ---- sampling ----
N_ROWS = 200
SEED   = 42

# ---- grading ----
RUBRIC = {
    0: "wrong or irrelevant",
    1: "mostly wrong, only slight overlap",
    2: "partially correct, misses key points",
    3: "mostly correct, minor omissions",
    4: "fully correct",
}
MAX_GRADE = max(RUBRIC)

THRESHOLD_MODE  = "fixed"          # fixed | auto_median
FIXED_THRESHOLD = 0.625            # = grade 2.5 / 4

# ---- generation ----
MAX_NEW_TOKENS = 220
BATCH_SIZE     = 25

# ---- schema ----
SET_EVAL_METRIC = "llm_as_a_judge"

# ---- models ----
GENERATORS = [
    {"name": "mistral", "repo": "mistralai/Mistral-7B-Instruct-v0.3"},   # ungated
    {"name": "llama",   "repo": "meta-llama/Llama-3.1-8B-Instruct"},     # GATED
    {"name": "gemma",   "repo": "google/gemma-2-9b-it", "attn": "eager"},# GATED
]

# independent of all three: different family, larger, ungated
JUDGE = {"name": "qwen3judge", "repo": "Qwen/Qwen3-14B"}

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,     # T4 has no bf16
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

SCHEMA_KEYS = [
    "id", "source", "category", "subcategory", "region", "language",
    "difficulty", "task_type", "question", "options", "answer",
    "explanation", "cultural_attr", "eval_metric",
]

os.makedirs(GEN_DIR, exist_ok=True)

print("Generators:")
for m in GENERATORS + [JUDGE]:
    cached = DRIVE_OK and os.path.isfile(
        os.path.join(CACHE_DIR, m["name"] + "_4bit", "config.json"))
    label = "JUDGE " if m is JUDGE else "      "
    print("  {}{:<11} {:<40} {}".format(
        label, m["name"], m["repo"], "cached" if cached else "will download"))
print("\ndevice:", "cuda" if torch.cuda.is_available() else "CPU (will be very slow)")

Generators:
        mistral     mistralai/Mistral-7B-Instruct-v0.3       cached
        llama       meta-llama/Llama-3.1-8B-Instruct         cached
        gemma       google/gemma-2-9b-it                     cached
  JUDGE qwen3judge  Qwen/Qwen3-14B                           cached

device: cuda


### Cell 4 - Load and sample

Samples 200 rows under a fixed seed, so the same rows return on every run -
which is what makes the resume logic safe across sessions.

Rows with an empty question or reference answer are dropped first; a judge
cannot grade against a missing reference.

The printed language split matters: roughly a third of this file is Hindi
literature, and that is the subset where judging is hardest. Cell 13 reports
Hindi and English scores separately so you can see whether the judge behaved
differently on them.

In [4]:
with open(INPUT_FILE, encoding="utf-8") as f:
    all_rows = [json.loads(line) for line in f]
print("Loaded {} rows from {}".format(len(all_rows), INPUT_FILE))


def usable(row):
    return (str(row.get("question") or "").strip()
            and len(str(row.get("answer") or "").split()) >= 3)


pool = [r for r in all_rows if usable(r)]
if len(pool) < len(all_rows):
    print("dropped {} rows with an empty question or reference".format(
        len(all_rows) - len(pool)))
assert len(pool) >= N_ROWS, "not enough usable rows"

random.seed(SEED)
sample = random.sample(pool, N_ROWS)

alen = sorted(len(str(r["answer"]).split()) for r in sample)
print("\nSampled {} rows".format(len(sample)))
print("  language : {}".format(dict(Counter(r["language"] for r in sample))))
print("  subjects : {}".format(dict(Counter(
    str(r["subcategory"]).split("_")[0] for r in sample).most_common(6))))
print("  reference answer words: min {}, median {}, max {}".format(
    alen[0], alen[len(alen) // 2], alen[-1]))

print("\n--- example row ---")
print("  Q: {}".format(" ".join(str(sample[0]["question"]).split())[:100]))
print("  A: {}".format(" ".join(str(sample[0]["answer"]).split())[:100]))

Loaded 946 rows from ncert_qa.jsonl
dropped 14 rows with an empty question or reference

Sampled 200 rows
  language : {'en': 131, 'hi': 69}
  subjects : {'ss': 54, 'hindi': 54, 'Eng': 25, 'english': 18, 'Hindi': 15, 'eng': 10}
  reference answer words: min 5, median 30, max 133

--- example row ---
  Q: How does having a good infrastructure help farming in Palampur?
  A: Good infrastructure like roads, electricity, and irrigation helps farming in Palampur. Roads make it


### Cell 5 - Answer prompt

The generators are asked to answer as an NCERT student would. Few-shot
examples come from **outside** the 200-row sample, so no scored row ever has
its reference answer shown to a generator.

The instruction asks for an answer in the same language as the question, since
a third of the file is Hindi and an English answer to a Hindi question would be
graded harshly for the wrong reason.

Two builders as usual: `build_completion` for base checkpoints, and
`build_chat_messages` for instruct ones. Cell 6 picks per model.

In [5]:
GEN_INSTRUCTIONS = (
    "You are answering questions from Indian NCERT school textbooks - Hindi "
    "literature, social science, history and civics.\n\n"
    "Answer the question directly and completely, the way a textbook answer "
    "key would. Answer in the SAME LANGUAGE as the question: Hindi questions "
    "get Hindi answers, English questions get English answers.\n\n"
    "Give only the answer - no preamble, no restating of the question."
)


def flat(text):
    return " ".join(str(text).split())


def pick_fewshot(k=2):
    used = {r["id"] for r in sample}
    cand = [r for r in pool
            if r["id"] not in used
            and 10 <= len(str(r["answer"]).split()) <= 40]
    random.Random(SEED + 1).shuffle(cand)
    # one Hindi and one English example if possible
    hi = [r for r in cand if r["language"] == "hi"][:1]
    en = [r for r in cand if r["language"] == "en"][:1]
    shots = (hi + en) or cand[:k]
    return shots[:k]


FEWSHOT = pick_fewshot()


def build_completion(question):
    text = GEN_INSTRUCTIONS + "\n"
    for r in FEWSHOT:
        text += "\nQuestion: {}\nAnswer: {}\n".format(
            flat(r["question"]), flat(r["answer"]))
    text += "\nQuestion: {}\nAnswer:".format(flat(question))
    return text


def build_chat_messages(question):
    msgs = [{"role": "system", "content": GEN_INSTRUCTIONS}]
    for r in FEWSHOT:
        msgs.append({"role": "user", "content": flat(r["question"])})
        msgs.append({"role": "assistant", "content": flat(r["answer"])})
    msgs.append({"role": "user", "content": flat(question)})
    return msgs


print("Few-shot examples ({}), all from OUTSIDE the sample:".format(len(FEWSHOT)))
for r in FEWSHOT:
    print("  {} [{}] {}".format(r["id"], r["language"], flat(r["question"])[:60]))

print("\n" + "=" * 66)
print(build_completion(sample[0]["question"]))
print("=" * 66)

Few-shot examples (2), all from OUTSIDE the sample:
  ncert_001301 [hi] रामचंद्र शुक्ल ने आधुनिक काल की सबसे प्रधान साहित्यिक घटना क
  ncert_000195 [en] The book that you gave me yesterday is an extraordinary (col

You are answering questions from Indian NCERT school textbooks - Hindi literature, social science, history and civics.

Answer the question directly and completely, the way a textbook answer key would. Answer in the SAME LANGUAGE as the question: Hindi questions get Hindi answers, English questions get English answers.

Give only the answer - no preamble, no restating of the question.

Question: रामचंद्र शुक्ल ने आधुनिक काल की सबसे प्रधान साहित्यिक घटना किसे बताया है?
Answer: रामचंद्र शुक्ल ने 'आधुनिक काल में गद्य का आविर्भाव' को सबसे प्रधान साहित्यिक घटना बताया है।

Question: The book that you gave me yesterday is an extraordinary (collage/college) of science fiction and mystery.
Answer: The book that you gave me yesterday is an extraordinary collage of science fiction and 

### Cell 6 - Shared model loading and answer generation

`load_model` implements download-once for **every** model including the judge:
if `models/<name>_4bit` exists in Drive it is loaded directly (already 4-bit, so
passing a fresh `BitsAndBytesConfig` would conflict and is omitted); otherwise
the repo is downloaded, quantised, and saved to Drive.

`trust_remote_code` stays off - repo-shipped modelling code is often written
against an older transformers API.

`clean_answer` strips the labels models prepend (`Answer:`, `Sure, ...`) and
cuts anything from a following `Question:` marker, which base models emit as
they continue the few-shot pattern.

In [6]:
def prompt_style(tokenizer):
    # base checkpoints have no chat template at all
    return "chat" if getattr(tokenizer, "chat_template", None) else "completion"


def cache_path(spec):
    return os.path.join(CACHE_DIR, spec["name"] + "_4bit")


def load_model(spec):
    cached     = cache_path(spec)
    from_drive = DRIVE_OK and os.path.isfile(os.path.join(cached, "config.json"))
    source     = cached if from_drive else spec["repo"]

    kwargs = {"device_map": "auto", "trust_remote_code": False}
    if from_drive:
        how = "Drive cache (already 4-bit)"
    else:
        kwargs["quantization_config"] = bnb_config
        how = "HuggingFace download -> 4-bit"
    if spec.get("attn"):
        kwargs["attn_implementation"] = spec["attn"]
        how += ", attn=" + spec["attn"]

    print("  loading {} [{}]".format(source, how))
    tokenizer = AutoTokenizer.from_pretrained(source)
    model = AutoModelForCausalLM.from_pretrained(source, **kwargs).eval()

    if not from_drive and DRIVE_OK:
        print("  saving 4-bit copy to {} (one time)...".format(cached))
        os.makedirs(cached, exist_ok=True)
        model.save_pretrained(cached)
        tokenizer.save_pretrained(cached)
        print("  saved - future runs skip the download")

    print("  ready | VRAM: {:.2f}GB | prompt style: {}".format(
        torch.cuda.memory_allocated() / 1e9, prompt_style(tokenizer)))
    return model, tokenizer


def unload(model, tokenizer):
    del model, tokenizer
    shutil.rmtree("/root/.cache/huggingface/hub/", ignore_errors=True)
    gc.collect()
    torch.cuda.empty_cache()


_LABEL = re.compile(r"^\s*(sure[,!]?\s*)?(the\s+)?answer\s*[:\-]\s*", re.I)


def clean_answer(text):
    t = (text or "").strip()
    t = re.split(r"\n\s*Question\s*:", t)[0]      # base model ran on
    t = _LABEL.sub("", t.strip())
    return " ".join(t.split())


@torch.no_grad()
def answer_question(model, tokenizer, question):
    if prompt_style(tokenizer) == "completion":
        text = build_completion(question)
    else:
        msgs = build_chat_messages(question)
        try:
            text = tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True)
        except Exception:
            # some templates (Gemma) reject a system role - fold it into the
            # first user turn rather than dropping the instructions
            merged = [dict(m) for m in msgs[1:]]
            merged[0]["content"] = msgs[0]["content"] + "\n\n" + merged[0]["content"]
            text = tokenizer.apply_chat_template(
                merged, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    n_in   = inputs["input_ids"].shape[1]
    out = model.generate(**inputs,
                         max_new_tokens=MAX_NEW_TOKENS,
                         do_sample=False,
                         pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    return clean_answer(tokenizer.decode(out[0][n_in:], skip_special_tokens=True))


for raw, want in [
    ("Answer: yeh sahi hai", "yeh sahi hai"),
    ("Sure, the answer: ok", "ok"),
    ("first line\nQuestion: next", "first line"),
    ("", ""),
]:
    got = clean_answer(raw)
    assert got == want, (raw, got, want)
print("Loading and generation functions defined")

Loading and generation functions defined


### Cell 7 - Run the three generators

Each generator answers all 200 questions, one model at a time - loaded, run,
unloaded - so peak VRAM stays near 6 GB. Every finished batch is appended to
`gen_progress/<model>.jsonl` before the next begins, so a disconnect costs at
most `BATCH_SIZE` rows and a completed model is skipped without loading.

The long cell: roughly **10-15 min per generator**, plus downloads on the first
run.

In [7]:
def run_generator(spec, rows):
    prog = os.path.join(GEN_DIR, spec["name"] + ".jsonl")

    done = {}
    if os.path.exists(prog):
        with open(prog, encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                done[item["id"]] = item
        print("  resuming - {}/{} already answered".format(len(done), len(rows)))

    remaining = [r for r in rows if r["id"] not in done]
    if not remaining:
        print("  {} already complete - skipping load".format(spec["name"]))
        return done

    model, tokenizer = load_model(spec)
    total_batches = (len(remaining) + BATCH_SIZE - 1) // BATCH_SIZE

    for start in range(0, len(remaining), BATCH_SIZE):
        batch = remaining[start:start + BATCH_SIZE]
        results = []
        for row in batch:
            ans = answer_question(model, tokenizer, row["question"])
            results.append({"id": row["id"], "answer": ans,
                            "n_words": len(ans.split())})
        with open(prog, "a", encoding="utf-8") as f:
            for item in results:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")
        done.update({i["id"]: i for i in results})
        print("  batch {}/{} saved - {}/{} rows".format(
            start // BATCH_SIZE + 1, total_batches, len(done), len(rows)))

    unload(model, tokenizer)
    print("  {} complete".format(spec["name"]))
    return done


answers = {}
for spec in GENERATORS:
    print("\n=== {} ===".format(spec["name"]))
    answers[spec["name"]] = run_generator(spec, sample)

print("\nAll generators done")


=== mistral ===
  resuming - 200/200 already answered
  mistral already complete - skipping load

=== llama ===
  resuming - 200/200 already answered
  llama already complete - skipping load

=== gemma ===
  resuming - 200/200 already answered
  gemma already complete - skipping load

All generators done


### Cell 8 - The judge

`grade()` builds a grading prompt containing the question, the reference
answer, and one model's answer, then reads the **logits over the rubric digits
`0`-`4` in a single forward pass**. Three consequences:

- The judge cannot produce unparseable output - the grade is chosen by argmax
  over five token ids, so there is no text to parse and no retries.
- It is far faster than generating a judgment: one forward pass per grade,
  so 600 grades take minutes rather than an hour.
- The softmax over those five digits gives a **probability-weighted expected
  grade**, a smooth 0-1 score rather than five discrete steps. That is what
  Cell 10 thresholds on, and it is why re-thresholding is free.

**Qwen3 needs `enable_thinking=False`.** It is a hybrid-reasoning model and its
chat template otherwise opens a `<think>` block, so the first generated token
would be the think tag rather than a digit. The call falls back gracefully if
your transformers version does not accept that argument.

An empty model answer is graded 0 without troubling the judge.

In [8]:
JUDGE_INSTRUCTIONS = (
    "You are grading answers to questions from Indian NCERT school textbooks "
    "(Hindi literature, social science, history, civics).\n\n"
    "You are given the question, the official reference answer, and a student "
    "answer. Grade how well the student answer matches the reference in "
    "FACTUAL CONTENT and MEANING. Ignore differences in wording, length, "
    "phrasing and writing style. An answer in a different language from the "
    "reference is fine if the content is correct.\n\n"
    "Grades:\n"
    + "\n".join("{} = {}".format(k, v) for k, v in sorted(RUBRIC.items()))
    + "\n\nReply with a single digit and nothing else."
)


def judge_user_turn(question, reference, answer):
    return ("Question:\n{}\n\nReference answer:\n{}\n\nStudent answer:\n{}"
            "\n\nGrade (0-{}):").format(
                flat(question), flat(reference), flat(answer), MAX_GRADE)


def judge_prompt(tokenizer, question, reference, answer):
    msgs = [{"role": "system", "content": JUDGE_INSTRUCTIONS},
            {"role": "user",   "content": judge_user_turn(question, reference, answer)}]
    try:
        # Qwen3 is a hybrid-reasoning model - without this it opens a <think>
        # block and the next token is a tag, not a digit
        return tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True,
            enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True)
    except Exception:
        merged = [{"role": "user",
                   "content": msgs[0]["content"] + "\n\n" + msgs[1]["content"]}]
        return tokenizer.apply_chat_template(
            merged, tokenize=False, add_generation_prompt=True)


def digit_token_ids(tokenizer):
    ids = {}
    for g in sorted(RUBRIC):
        variants = set()
        for form in (str(g), " " + str(g)):
            enc = tokenizer.encode(form, add_special_tokens=False)
            if enc:
                variants.add(enc[0])
        ids[g] = sorted(variants)
    return ids


@torch.no_grad()
def grade(model, tokenizer, tok_ids, question, reference, answer):
    if not str(answer).strip():
        return 0, 0.0                      # nothing to grade

    text   = judge_prompt(tokenizer, question, reference, answer)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    logits = model(**inputs).logits[0, -1]

    per_digit = torch.tensor(
        [max(logits[i].item() for i in tok_ids[g]) for g in sorted(RUBRIC)])
    probs = torch.softmax(per_digit, dim=0)

    grades   = torch.tensor([float(g) for g in sorted(RUBRIC)])
    argmax   = int(grades[int(torch.argmax(probs))].item())
    expected = float((probs * grades).sum().item()) / MAX_GRADE   # 0..1
    return argmax, expected

print("Judge functions defined")

Judge functions defined


### Cell 9 - Run the judge

Loads Qwen3-14B **once** and grades all 600 answers (200 rows x 3 generators),
appending each batch to `judge_progress.jsonl` keyed by row and model, so a
disconnect costs at most one batch.

Grades are stored raw - both the argmax digit and the smooth expected score.
Nothing is thresholded here, which is what lets Cell 10 be re-run freely.

In [10]:
import os
import shutil

def run_judge(rows, answers):
    # Clear potentially corrupted cache for the judge model to force re-download
    judge_cache_path = os.path.join(CACHE_DIR, JUDGE["name"] + "_4bit")
    if DRIVE_OK and os.path.isdir(judge_cache_path):
        print(f"Attempting to clear potentially corrupted cache for {JUDGE['name']} at {judge_cache_path}")
        try:
            shutil.rmtree(judge_cache_path)
            print(f"Cleared cache for {JUDGE['name']}. Model will be re-downloaded.")
        except Exception as e:
            print(f"Failed to clear cache: {e}. Proceeding with existing cache.")

    done = {}
    if os.path.exists(JUDGE_FILE):
        with open(JUDGE_FILE, encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                done[(item["id"], item["model"])] = item
        print("resuming - {} grades already done".format(len(done)))

    todo = [(r, s["name"]) for r in rows for s in GENERATORS
            if (r["id"], s["name"]) not in done]
    if not todo:
        print("judging already complete - skipping load")
        return done
    print("grades to compute: {}".format(len(todo)))

    model, tokenizer = load_model(JUDGE)
    tok_ids = digit_token_ids(tokenizer)

    total_batches = (len(todo) + BATCH_SIZE - 1) // BATCH_SIZE
    for start in range(0, len(todo), BATCH_SIZE):
        batch, results = todo[start:start + BATCH_SIZE], []
        for row, mname in batch:
            hyp = answers[mname][row["id"]]["answer"]
            g, s = grade(model, tokenizer, tok_ids,
                         row["question"], row["answer"], hyp)
            results.append({"id": row["id"], "model": mname,
                            "grade": g, "score": round(s, 4)})
        with open(JUDGE_FILE, "a", encoding="utf-8") as f:
            for item in results:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")
        done.update({(i["id"], i["model"]): i for i in results})
        mean = sum(v["score"] for v in done.values()) / len(done)
        print("  batch {}/{} saved - {}/{} grades | mean score {:.1%}".format(
            start // BATCH_SIZE + 1, total_batches,
            len(done), len(todo) + (len(done) - len(todo)), mean))

    unload(model, tokenizer)
    print("judging complete")
    return done


grades = run_judge(sample, answers)
print("\ntotal grades: {}".format(len(grades)))

Attempting to clear potentially corrupted cache for qwen3judge at /drive/MyDrive/models/qwen3judge_4bit
Cleared cache for qwen3judge. Model will be re-downloaded.
grades to compute: 600
  loading Qwen/Qwen3-14B [HuggingFace download -> 4-bit]


Loading weights:   0%|          | 0/443 [00:00<?, ?it/s]

  saving 4-bit copy to /drive/MyDrive/models/qwen3judge_4bit (one time)...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved - future runs skip the download
  ready | VRAM: 9.97GB | prompt style: chat
  batch 1/24 saved - 25/25 grades | mean score 53.5%
  batch 2/24 saved - 50/50 grades | mean score 56.9%
  batch 3/24 saved - 75/75 grades | mean score 61.7%
  batch 4/24 saved - 100/100 grades | mean score 62.3%
  batch 5/24 saved - 125/125 grades | mean score 63.1%
  batch 6/24 saved - 150/150 grades | mean score 64.1%
  batch 7/24 saved - 175/175 grades | mean score 64.3%
  batch 8/24 saved - 200/200 grades | mean score 64.5%
  batch 9/24 saved - 225/225 grades | mean score 65.1%
  batch 10/24 saved - 250/250 grades | mean score 64.5%
  batch 11/24 saved - 275/275 grades | mean score 64.1%
  batch 12/24 saved - 300/300 grades | mean score 64.1%
  batch 13/24 saved - 325/325 grades | mean score 64.4%
  batch 14/24 saved - 350/350 grades | mean score 63.8%
  batch 15/24 saved - 375/375 grades | mean score 63.9%
  batch 16/24 saved - 400/400 grades | mean score 64.5%
  batch 17/24 saved - 425/425 grade

### Cell 10 - Find the threshold

Prints the grade distribution per generator and pooled, then applies
`THRESHOLD_MODE`.

The default is `"fixed"` at **0.625**, which on the 0-4 rubric means an
expected grade above 2.5 - better than "partially correct, misses key points".
That is a rubric-anchored choice, and it is more meaningful here than a median,
because the rubric already has an interpretable scale. `"auto_median"` is
available if you would rather have a balanced split.

The sensitivity table shows what each threshold does, and warnings fire if the
chosen value empties a difficulty band or sits at an extreme.

In [11]:
pooled = sorted(v["score"] for v in grades.values())

def pct(p):
    return pooled[min(len(pooled) - 1, int(p * len(pooled)))]

print("judge score per generator (0-1, from the 0-{} rubric):".format(MAX_GRADE))
print("  {:<10} {:>7} {:>7} {:>7} {:>7}".format("model", "p25", "median", "p75", "mean"))
for s in GENERATORS:
    v = sorted(grades[(r["id"], s["name"])]["score"] for r in sample)
    print("  {:<10} {:>6.1%} {:>7.1%} {:>7.1%} {:>7.1%}".format(
        s["name"], v[len(v) // 4], v[len(v) // 2], v[3 * len(v) // 4],
        sum(v) / len(v)))
print("  {:<10} {:>6.1%} {:>7.1%} {:>7.1%} {:>7.1%}".format(
    "POOLED", pct(.25), pct(.50), pct(.75), sum(pooled) / len(pooled)))

print("\nargmax grade distribution:")
for s in GENERATORS:
    c = Counter(grades[(r["id"], s["name"])]["grade"] for r in sample)
    print("  {:<10} {}".format(s["name"], {g: c.get(g, 0) for g in sorted(RUBRIC)}))


def difficulty_at(th):
    out = Counter()
    for r in sample:
        votes = sum(grades[(r["id"], s["name"])]["score"] >= th for s in GENERATORS)
        out["Easy" if votes == 3 else ("Medium" if votes == 2 else "Hard")] += 1
    return out


if THRESHOLD_MODE == "fixed":
    THRESHOLD = FIXED_THRESHOLD
    why = "fixed - expected grade above {:.1f} of {}".format(
        FIXED_THRESHOLD * MAX_GRADE, MAX_GRADE)
elif THRESHOLD_MODE == "auto_median":
    THRESHOLD = pct(.50)
    why = "median of all pooled judge scores"
else:
    raise ValueError("unknown THRESHOLD_MODE: " + str(THRESHOLD_MODE))

print("\nsensitivity - what each threshold would produce:")
print("  {:>9}  {:>6} {:>7} {:>6}".format("threshold", "Easy", "Medium", "Hard"))
for th in sorted(set(round(x, 3) for x in
                     [.25, .375, .5, .625, .75, .875, round(THRESHOLD, 3)])):
    d = difficulty_at(th)
    tag = "  <- CHOSEN" if abs(th - round(THRESHOLD, 3)) < 1e-9 else ""
    print("  {:>9.3f}  {:>6} {:>7} {:>6}{}".format(
        th, d.get("Easy", 0), d.get("Medium", 0), d.get("Hard", 0), tag))

print("\nTHRESHOLD = {:.3f}  ({})".format(THRESHOLD, why))
_b = difficulty_at(THRESHOLD)
if min(_b.get(k, 0) for k in ("Easy", "Medium", "Hard")) == 0:
    print("  WARNING: one difficulty band is empty - the split carries little")
    print("  information. Pick another value from the table above.")

judge score per generator (0-1, from the 0-4 rubric):
  model          p25  median     p75    mean
  mistral     50.0%   50.0%  100.0%   61.5%
  llama       50.0%   50.0%   75.0%   57.1%
  gemma       50.0%   75.0%  100.0%   71.3%
  POOLED      50.0%   51.9%   99.8%   63.3%

argmax grade distribution:
  mistral    {0: 26, 1: 7, 2: 82, 3: 21, 4: 64}
  llama      {0: 23, 1: 6, 2: 97, 3: 37, 4: 37}
  gemma      {0: 16, 1: 3, 2: 56, 3: 45, 4: 80}

sensitivity - what each threshold would produce:
  threshold    Easy  Medium   Hard
      0.250     156      28     16
      0.375     147      34     19
      0.500     120      44     36
      0.625      56      36    108  <- CHOSEN
      0.750      44      39    117
      0.875      27      34    139

THRESHOLD = 0.625  (fixed - expected grade above 2.5 of 4)


### Cell 11 - Apply the threshold and write the schema

Each generator votes 1 where its judge score clears `THRESHOLD`; the votes sum
into Easy / Medium / Hard as in every other split.

Output rows are rebuilt key-by-key from `SCHEMA_KEYS`, so the file carries
exactly the 14 IndicSample fields in schema order. `difficulty` and
`eval_metric` are the only values that change. Every model answer and grade
goes to the audit file, along with the judge's identity - without that recorded
somewhere, an `llm_as_a_judge` score is not reproducible.

In [12]:
def get_difficulty(votes):
    score = sum(votes)
    if score == 3:
        return "Easy"
    elif score == 2:
        return "Medium"
    else:
        return "Hard"


final_results, audit = [], []

for row in sample:
    scores = [grades[(row["id"], s["name"])]["score"] for s in GENERATORS]
    votes  = [int(x >= THRESHOLD) for x in scores]
    difficulty = get_difficulty(votes)

    enriched = {**row, "difficulty": difficulty}
    if SET_EVAL_METRIC:
        enriched["eval_metric"] = SET_EVAL_METRIC
    final_results.append({k: enriched.get(k) for k in SCHEMA_KEYS})

    audit.append({
        "id":         row["id"],
        "difficulty": difficulty,
        "votes":      votes,
        "scores":     [round(x, 4) for x in scores],
        "grades":     [grades[(row["id"], s["name"])]["grade"] for s in GENERATORS],
        "threshold":  round(THRESHOLD, 4),
        "judge":      JUDGE["repo"],
        "rubric":     RUBRIC,
        "language":   row["language"],
        "question":   " ".join(str(row["question"]).split()),
        "reference":  " ".join(str(row["answer"]).split()),
        "answers":    {s["name"]: answers[s["name"]][row["id"]]["answer"]
                       for s in GENERATORS},
    })

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for item in final_results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
with open(AUDIT_FILE, "w", encoding="utf-8") as f:
    for item in audit:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Saved -> {} ({} rows)".format(OUTPUT_FILE, len(final_results)))
print("Audit -> {}  (judge: {})".format(AUDIT_FILE, JUDGE["repo"]))
print("threshold used: {:.3f}".format(THRESHOLD))

Saved -> ncert_qa_difficulty.jsonl (200 rows)
Audit -> ncert_qa_audit.jsonl  (judge: Qwen/Qwen3-14B)
threshold used: 0.625


### Cell 12 - Verify and report

Checks before trusting the file:

1. **Schema** - all 14 keys in order, no nulls in `difficulty`, `eval_metric`
   updated.
2. **Difficulty distribution.**
3. **Per-generator mean judge score**, plus the share of empty answers - an
   empty answer is graded 0 and votes Hard for a formatting reason, not a
   knowledge one.
4. **Hindi vs English breakdown.** This is the check specific to this file: a
   third of it is Hindi literature, and if the Hindi scores are dramatically
   lower across *all three* generators, that may reflect the judge's Hindi
   ability rather than the questions' difficulty. Compare before trusting the
   Hindi labels.

The sample rows print the question, reference, and all three answers with their
grades - the quickest way to sanity-check that the judge is behaving.

In [13]:
bad_keys = [r["id"] for r in final_results if list(r.keys()) != SCHEMA_KEYS]
missing  = [r["id"] for r in final_results if r["difficulty"] is None]
metric   = Counter(r["eval_metric"] for r in final_results)
print("Schema check : {} rows | wrong keys: {} | null difficulty: {}".format(
    len(final_results), len(bad_keys), len(missing)))
print("eval_metric  : {}".format(dict(metric)))

dist  = Counter(r["difficulty"] for r in final_results)
total = len(final_results)
print("\nDifficulty distribution (threshold {:.3f}):".format(THRESHOLD))
for level in ["Easy", "Medium", "Hard"]:
    n = dist.get(level, 0)
    print("  {:<7}: {:4d}  ({:.1f}%)".format(level, n, n / total * 100))

print("\nPer-generator:")
for s in GENERATORS:
    v = [grades[(r["id"], s["name"])]["score"] for r in sample]
    empty = sum(1 for r in sample
                if not answers[s["name"]][r["id"]]["answer"].strip())
    flag = "  <- empty answers, a formatting problem" if empty / total > 0.05 else ""
    print("  {:<10} mean score {:.1%} | empty {:>3}/{}{}".format(
        s["name"], sum(v) / len(v), empty, total, flag))

print("\nHindi vs English (mean judge score):")
for lang in sorted({r["language"] for r in sample}):
    rows_l = [r for r in sample if r["language"] == lang]
    line = "  {:<4} n={:3d}".format(lang, len(rows_l))
    for s in GENERATORS:
        v = [grades[(r["id"], s["name"])]["score"] for r in rows_l]
        line += "  {}={:.1%}".format(s["name"][:4], sum(v) / len(v))
    d = Counter(r["difficulty"] for r in final_results
                if r["language"] == lang)
    line += "  | E{} M{} H{}".format(
        d.get("Easy", 0), d.get("Medium", 0), d.get("Hard", 0))
    print(line)

print("\n--- 2 sample rows ---")
for a in audit[:2]:
    print("\n  {} [{}] {} | grades={} scores={}".format(
        a["id"], a["language"], a["difficulty"], a["grades"], a["scores"]))
    print("    Q  :", a["question"][:92])
    print("    ref:", a["reference"][:92])
    for k, v in a["answers"].items():
        print("    {:<7}:".format(k), (v or "<empty>")[:92])

Schema check : 200 rows | wrong keys: 0 | null difficulty: 0
eval_metric  : {'llm_as_a_judge': 200}

Difficulty distribution (threshold 0.625):
  Easy   :   56  (28.0%)
  Medium :   36  (18.0%)
  Hard   :  108  (54.0%)

Per-generator:
  mistral    mean score 61.5% | empty   0/200
  llama      mean score 57.1% | empty   0/200
  gemma      mean score 71.3% | empty   0/200

Hindi vs English (mean judge score):
  en   n=131  mist=74.2%  llam=65.1%  gemm=80.5%  | E56 M30 H45
  hi   n= 69  mist=37.6%  llam=42.0%  gemm=53.8%  | E0 M6 H63

--- 2 sample rows ---

  ncert_002087 [en] Medium | grades=[4, 2, 3] scores=[1.0, 0.5298, 0.7501]
    Q  : How does having a good infrastructure help farming in Palampur?
    ref: Good infrastructure like roads, electricity, and irrigation helps farming in Palampur. Roads
    mistral: Having a good infrastructure helps farming in Palampur by facilitating the transportation of
    llama  : पालमपुर में अच्छी ढांचे की उपस्थिति कृषि को कई तरह से मदद करती है। यहा

In [14]:
try:
    from google.colab import files
    files.download(OUTPUT_FILE)
except ImportError:
    print("not downloaded")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>